# ByteNet - FFT-75 Benchmark

This notebook runs ByteNet on the two Kaggle inputs you uploaded:

- `FFT_75_512_1`
- `FFT_75_4096_1`

Each dataset already contains `train.npz`, `val.npz`, and `test.npz`.

Workflow:
1. Install dependencies
2. Clone the repo
3. Locate the Kaggle inputs
4. Train and evaluate ByteNet on 512B
5. Train and evaluate ByteNet on 4096B
6. Compare the results with the paper targets from `benchmarks/ByteNet/docs/architecture.md`


## Cell 1 - Install Dependencies

In [ ]:
import subprocess
import sys


def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])


pip('pyyaml>=6.0')
pip('scikit-learn>=1.5')
pip('pandas>=2.2')
pip('numpy>=1.26')
pip('matplotlib>=3.9')
pip('seaborn>=0.13')
pip('tqdm>=4.66')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM    : {props.total_memory / 1e9:.1f} GB')


## Cell 2 - Clone Repo

In [ ]:
import gc
import os
import shutil
from pathlib import Path

WORKING = Path('/kaggle/working')
REPO_DIR = WORKING / 'deepcarv'
BRANCH_NAME = 'eval-ByteNet'
GITHUB_TOKEN = ''

if not REPO_DIR.exists() or not (REPO_DIR / 'src').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    if GITHUB_TOKEN:
        clone_url = f'https://{GITHUB_TOKEN}@github.com/yuvnahr/deepcarv.git'
    else:
        clone_url = 'https://github.com/yuvnahr/deepcarv.git'
    os.system(f'git clone --depth 1 --branch {BRANCH_NAME} {clone_url} {REPO_DIR}')
else:
    print('Repo already present:', REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.environ['KAGGLE_RUNTIME'] = '1'

print('sys.path[0]:', sys.path[0])
print('src exists :', (REPO_DIR / 'src').exists())
print('benchmarks :', (REPO_DIR / 'benchmarks' / 'ByteNet').exists())

gc.collect()


## Cell 3 - Locate Kaggle Datasets

This cell reads the two mounted Kaggle inputs and normalizes them into the benchmark layout expected by the training script:

- `/kaggle/working/data/FFT-75/512/{train,val,test}.npz`
- `/kaggle/working/data/FFT-75/4096/{train,val,test}.npz`


In [ ]:
import gc
import json
from pathlib import Path

import numpy as np

from src.data.verify_dataset import verify_dataset

KAGGLE_INPUT = Path('/kaggle/input')
CANONICAL_ROOT = WORKING / 'data' / 'FFT-75'
CANONICAL_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = {
    512: KAGGLE_INPUT / 'FFT_75_512_1',
    4096: KAGGLE_INPUT / 'FFT_75_4096_1',
}
SPLITS = ('train', 'val', 'test')


def _load_npz_arrays(path: Path) -> tuple[np.ndarray, np.ndarray]:
    with np.load(path, allow_pickle=False) as data:
        keys = {k.lower(): k for k in data.files}
        if 'x' not in keys or 'y' not in keys:
            raise ValueError(f'{path.name} must contain x/y arrays, found {data.files}')
        return np.asarray(data[keys['x']]), np.asarray(data[keys['y']])


def _write_normalized_split(src: Path, dst: Path, fragment_size: int) -> None:
    X, y = _load_npz_arrays(src)
    if X.ndim == 1 and X.size % fragment_size == 0:
        X = X.reshape(-1, fragment_size)
    if X.ndim != 2 or X.shape[1] != fragment_size:
        raise ValueError(f'{src}: expected X shape [N, {fragment_size}], got {X.shape}')
    if y.ndim != 1:
        y = y.reshape(-1)
    if len(X) != len(y):
        raise ValueError(f'{src}: len(X)={len(X)} != len(y)={len(y)}')
    np.savez_compressed(dst, x=X, y=y)


def prepare_dataset(fragment_size: int, kaggle_root: Path) -> Path:
    if not kaggle_root.exists():
        raise FileNotFoundError(f'Missing Kaggle input folder: {kaggle_root}')

    target_dir = CANONICAL_ROOT / str(fragment_size)
    target_dir.mkdir(parents=True, exist_ok=True)

    print(f'Preparing {fragment_size}B dataset from: {kaggle_root}')
    for split in SPLITS:
        src = kaggle_root / f'{split}.npz'
        if not src.exists():
            raise FileNotFoundError(f'Missing split file: {src}')
        dst = target_dir / f'{split}.npz'
        _write_normalized_split(src, dst, fragment_size)
        print(f'  {split}: {dst}')

    return target_dir


for fragment_size, kaggle_root in DATASETS.items():
    prepare_dataset(fragment_size, kaggle_root)

print()
print('Canonical layout:')
for fragment_size in (512, 4096):
    for split in SPLITS:
        path = CANONICAL_ROOT / str(fragment_size) / f'{split}.npz'
        print(f'  {fragment_size}/{split}.npz -> {path.exists()}')

for fragment_size in (512, 4096):
    print()
    print(f'Verifying {fragment_size}B dataset')
    verify_dataset(data_dir=CANONICAL_ROOT, fragment_size=fragment_size)

print()
gc.collect()


## Cell 4 - Run ByteNet Training and Evaluation

In [ ]:
import gc
import json

import torch

from benchmarks.ByteNet.scripts.evaluate import main as bytenet_eval_main
from benchmarks.ByteNet.scripts.train import main as bytenet_train_main

CKPT_DIR = WORKING / 'checkpoints'
OUTPUTS_DIR = WORKING / 'outputs'
LOGS_DIR = WORKING / 'logs'
for d in (CKPT_DIR, OUTPUTS_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

VARIANT = 'bytenet_resnet'  # 'bytenet_resnet' | 'bytenet_former'
PAPER_TARGETS = {
    'bytenet_resnet': {512: 0.710, 4096: 0.821},
    'bytenet_former': {512: 0.732, 4096: 0.819},
}
RESULTS = {}


def run_bytenet(fragment_size: int, train_batch_size: int, eval_batch_size: int = 256) -> dict:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    best_ckpt = CKPT_DIR / f'best_bytenet_{VARIANT}_{fragment_size}b.pt'
    run_dir = OUTPUTS_DIR / f'bytenet_{VARIANT}_{fragment_size}b'
    eval_dir = OUTPUTS_DIR / f'bytenet_{VARIANT}_{fragment_size}b_eval'

    bytenet_train_main([
        '--data_dir', str(CANONICAL_ROOT),
        '--fragment_size', str(fragment_size),
        '--variant', VARIANT,
        '--epochs', '50',
        '--batch_size', str(train_batch_size),
        '--lr', '5e-4',
        '--seed', '42',
        '--checkpoint_path', str(best_ckpt),
        '--run_dir', str(run_dir),
    ])

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    bytenet_eval_main([
        '--checkpoint', str(best_ckpt),
        '--data_dir', str(CANONICAL_ROOT),
        '--fragment_size', str(fragment_size),
        '--variant', VARIANT,
        '--out_dir', str(eval_dir),
        '--batch_size', str(eval_batch_size),
        '--seed', '42',
    ])

    with open(eval_dir / 'metrics.json', encoding='utf-8') as f:
        metrics = json.load(f)

    target = PAPER_TARGETS[VARIANT][fragment_size]
    RESULTS[fragment_size] = {
        'fragment_size': fragment_size,
        'variant': VARIANT,
        'accuracy': metrics.get('accuracy', 0.0),
        'macro_f1': metrics.get('macro_f1', 0.0),
        'weighted_f1': metrics.get('weighted_f1', 0.0),
        'paper_target_accuracy': target,
        'delta_accuracy': metrics.get('accuracy', 0.0) - target,
        'checkpoint': str(best_ckpt),
        'run_dir': str(run_dir),
        'eval_dir': str(eval_dir),
    }

    print()
    print(f'=== {fragment_size}B Results ===')
    print(f"  Accuracy   : {metrics.get('accuracy', 0.0):.4f}")
    print(f"  Macro F1   : {metrics.get('macro_f1', 0.0):.4f}")
    print(f"  Weighted F1: {metrics.get('weighted_f1', 0.0):.4f}")
    print(f'  Paper acc  : {target:.3f}')
    print(f"  Delta      : {metrics.get('accuracy', 0.0) - target:+.4f}")

    return RESULTS[fragment_size]


## Cell 5 - Train and Evaluate 512B

In [ ]:
run_bytenet(fragment_size=512, train_batch_size=512, eval_batch_size=256)


## Cell 6 - Train and Evaluate 4096B

In [ ]:
run_bytenet(fragment_size=4096, train_batch_size=128, eval_batch_size=256)


## Cell 7 - Paper Comparison and Bundle

In [ ]:
import json
import shutil

comparison_path = OUTPUTS_DIR / f'bytenet_{VARIANT}_comparison.json'
comparison_path.write_text(json.dumps(RESULTS, indent=2, sort_keys=True) + '\n', encoding='utf-8')

print()
print('=== Paper Comparison ===')
print('Size   Accuracy   Paper     Delta     Macro F1   Weighted F1')
for size in (512, 4096):
    result = RESULTS.get(size)
    if not result:
        continue
    print(
        f"{size:<6} {result['accuracy']:.4f}   {result['paper_target_accuracy']:.3f}  "
        f"{result['delta_accuracy']:+.4f}   {result['macro_f1']:.4f}   {result['weighted_f1']:.4f}"
    )

bundle_dir = WORKING / f'ByteNet_{VARIANT}_bundle'
bundle_dir.mkdir(parents=True, exist_ok=True)

for size in (512, 4096):
    result = RESULTS[size]
    eval_dir = Path(result['eval_dir'])
    run_dir = Path(result['run_dir'])
    ckpt = Path(result['checkpoint'])

    for fname in (
        'metrics.json',
        'summary.json',
        'predictions.csv',
        'confusion_matrix.csv',
        'per_class_metrics.csv',
        'classification_report.txt',
    ):
        src = eval_dir / fname
        if src.exists():
            shutil.copy2(src, bundle_dir / f'{size}_{fname}')

    curve = run_dir / 'training_curves.png'
    if curve.exists():
        shutil.copy2(curve, bundle_dir / f'{size}_training_curves.png')

    if ckpt.exists():
        shutil.copy2(ckpt, bundle_dir / ckpt.name)

shutil.copy2(comparison_path, bundle_dir / comparison_path.name)

zip_path = WORKING / f'ByteNet_{VARIANT}_bundle.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', bundle_dir)

print()
print(f'Bundle: {zip_path}')
print(f'Size  : {zip_path.stat().st_size / 1e6:.1f} MB')
print('Files :')
for f in sorted(bundle_dir.iterdir()):
    print(f'  {f.name}')
